In [0]:
# Databricks notebook source
import os
import yaml
from pyspark.sql import functions as F
from pyspark.sql.window import Window

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
caminho_config = os.path.join(repo_root, "config", "series_sgs.yaml")
with open(caminho_config, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

catalogo = config["catalogo"]
tabela_silver = f"{catalogo}.silver.sgs_series_tratadas"
tabela_gold_fato = f"{catalogo}.gold.fato_series_sgs"
tabela_gold_pivot = f"{catalogo}.gold.indicadores_pivot"

df_silver = spark.read.table(tabela_silver)

# COMMAND ----------

# 1. Tabela Fato Gold (com cálculo de valor anterior e variação absoluta)
janela_serie = Window.partitionBy("id_serie").orderBy("data_referencia")

df_gold_fato = (
    df_silver
    .withColumn("valor_anterior", F.lag("valor", 1).over(janela_serie))
    .withColumn("variacao_absoluta", F.col("valor") - F.col("valor_anterior"))
    .withColumn("ano", F.year("data_referencia"))
    .withColumn("mes", F.month("data_referencia"))
    .withColumn("data_atualizacao", F.current_timestamp())
)

(
    df_gold_fato.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_gold_fato)
)

# COMMAND ----------

# 2. Tabela Gold Pivotada (uma coluna por código de série para cruzamento analítico)
df_gold_pivot = (
    df_silver
    .withColumn("coluna_serie", F.concat(F.lit("serie_"), F.col("id_serie")))
    .groupBy("data_referencia")
    .pivot("coluna_serie")
    .agg(F.first("valor"))
    .orderBy("data_referencia")
)

(
    df_gold_pivot.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_gold_pivot)
)